In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler, StandardScaler, MinMaxScaler, OneHotEncoder, PolynomialFeatures
from sklearn.model_selection import train_test_split
import re
from sklearn.ensemble import RandomForestRegressor

from sklearn.linear_model import LinearRegression, Ridge, RidgeCV, LassoCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error

from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

In [2]:
df = pd.read_csv("https://drive.google.com/uc?export=download&id=1D5gl2EZtevQCf9AgRR3MWtkmOefu_qnd")

In [ ]:
df = df.drop(columns = ["Unnamed: 0"])

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.describe()

# Data Cleaning

## Feature Engineering

# Normalising Feature Names

In [ ]:
df.rename(columns= {"Economy_status_Developed" : "economic_status"}, inplace = True)
df.head()

In [ ]:
df.columns.tolist()

In [ ]:
df.columns = df.columns.str.lower().str.strip()


In [ ]:
df.columns.tolist()

In [ ]:
df["thinness_five_nineteen_years"] = round(((df["thinness_ten_nineteen_years"] + df["thinness_five_nine_years"]) / 2), 2)

In [ ]:
df.head()

In [ ]:
df = df.drop(columns = ["economy_status_developing", "thinness_ten_nineteen_years", "thinness_five_nine_years"])

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
numeric_columns = list(df.select_dtypes(include = ["number"]).columns)
numeric_columns.remove("economic_status")

In [ ]:
numeric_columns

In [ ]:
len(numeric_columns)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


def plot_numeric_boxplots(df, numeric_columns):
    fig, axes = plt.subplots(8,2, figsize=(16,30)) 
    axes = axes.flatten()

    colour_palette = ["#0B132B","#1C46C0","#00E5FF","#6A67CE","#FF4FD8"]

    for feature in range(len(numeric_columns)):
        sns.boxplot(
            x=df[numeric_columns[feature]],
            ax=axes[feature],
            color=colour_palette[(feature // 2) % len(colour_palette)])  # cycles back to the first colour
        axes[feature].set_xlabel(numeric_columns[feature].replace("_", " ").title(),fontweight="bold") # for presentation purposes!

    # fig.delaxes(axes[-1]) # 17 features 18 graphs, this deletes that extra graph

    plt.suptitle("Plot of Numeric Features\n\n", fontweight="bold", fontsize=24)
    plt.tight_layout()
    plt.show()

## Plot Exploring Outliers Within Each Numeric Feature

In [ ]:
plot_numeric_boxplots(df, numeric_columns)

## Winsorisation -- outlier capping

In [ ]:
def cap_outliers(df, numeric_columns):
    df_clean = df.copy()

    for column in numeric_columns:
        Q1 = df_clean[column].quantile(0.25)
        Q3 = df_clean[column].quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        df_clean[column] = df_clean[column].clip(lower_bound, upper_bound)

    return df_clean

In [ ]:
df = cap_outliers(df, numeric_columns)

## Plot After Clipping Outliers

In [ ]:
plot_numeric_boxplots(df, numeric_columns)

In [ ]:
df_eda = df.copy()
df_eda = df_eda.drop(columns = ["country", "region"])

In [ ]:
df_eda.corr()

In [ ]:
plt.figure(figsize = (16, 12))
sns.heatmap(df_eda.corr(), annot = True, fmt = ".3f")
plt.tight_layout()
plt.show()

In [ ]:
plt.hist(df["life_expectancy"], color= 'coral', edgecolor = "black")
plt.title("Life Expectancy Distribution ")
plt.xlabel("Life Expectancy")

In [ ]:
def count_plot(data, x_axis=None ,y_axis=None, hue=None, rotation=0, top=None, ax=None,):
    """Draw a Seaborn countplot ordered by category frequency with bar value labels."""

    if x_axis is None and y_axis is None:
        raise ValueError("Either 'x_axis' or 'y_axis' must be specified.")

    target_col = x_axis if x_axis is not None else y_axis
    order = data[target_col].value_counts().iloc[:top].index # choose the amount of entries you'd like to see

    graph = sns.countplot(
        data=data,
        x=x_axis,
        y=y_axis,
        hue=hue,
        order=order,
        ax=ax,
    )

    for container in graph.containers:
        graph.bar_label(container)

    if x_axis is not None:
        graph.tick_params(axis="x", rotation=rotation)

    return graph

# Feature Selection

## VIF

In [ ]:
df.columns.to_list()

In [ ]:
cols_to_check = [
 'year',
 'infant_deaths',
 'under_five_deaths',
 'adult_mortality',
 'alcohol_consumption',
 'hepatitis_b',
 'measles',
 'bmi',
 'polio',
 'diphtheria',
 'incidents_hiv',
 'gdp_per_capita',
 'population_mln',
 'schooling',
 'thinness_five_nineteen_years']

#  'economic_status'

In [ ]:
df_with_const = add_constant(df[cols_to_check])

vif_series = pd.Series(
    [
        variance_inflation_factor(df_with_const.values, i)
        for i in range(df_with_const.shape[1])
    ],
    index=df_with_const.columns,
).drop("const")

print(vif_series.sort_values(ascending=False))

In [ ]:
cols_to_drop = vif_series[vif_series > 5].index.to_list()

In [ ]:
cols_to_drop

In [ ]:
# df = df.drop(columns = cols_to_drop)

In [ ]:
df.head()

In [ ]:
df["region"].unique()

In [ ]:
X = df.drop(columns = ["life_expectancy", "country"])
y = df["life_expectancy"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42) # test out 0.3 after

# Encoding

In [ ]:
X_train_encoded = pd.get_dummies(X_train, columns=["region"], drop_first=True, dtype=int)

X_test_encoded = pd.get_dummies(X_test, columns=["region"], drop_first=True, dtype=int)

X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join="left", axis=1, fill_value=0)

## Polynomial Features

In [ ]:
poly = PolynomialFeatures(degree=2, include_bias=False)

X_train_poly = poly.fit_transform(X_train_encoded)
X_test_poly = poly.transform(X_test_encoded)

feature_names = poly.get_feature_names_out(X_train_encoded.columns)

X_train_encoded_poly = pd.DataFrame(X_train_poly, columns=feature_names, index=X_train_encoded.index)

X_test_encoded_poly = pd.DataFrame(X_test_poly, columns=feature_names, index=X_test_encoded.index)

# Scaling

In [ ]:
scaler = StandardScaler()

X_train_encoded_poly_scaled = pd.DataFrame(scaler.fit_transform(X_train_encoded_poly), columns=X_train_encoded_poly.columns, index=X_train_encoded_poly.index)

X_test_encoded_poly_scaled = pd.DataFrame(scaler.transform(X_test_encoded_poly), columns=X_test_encoded_poly.columns, index=X_test_encoded_poly.index)

# Stepwise Selection

In [ ]:
def stepwise_selection(X, y, threshold_in=0.01, threshold_out=0.05, verbose=True):
    # The function checks p-values (statistical significance of features) - lower is better
    included = []  # list of features we keep
    while True:
        changed = False

        # forward step: try adding each excluded feature and pick the most significant one
        excluded = list(set(X.columns) - set(included))
        new_pval = pd.Series(index=excluded, dtype='float64')
        for new_column in excluded:
            model = sm.OLS(y, sm.add_constant(pd.DataFrame(X[included + [new_column]]))).fit()
            new_pval[new_column] = model.pvalues[new_column]
        best_pval = new_pval.min()
        # add the feature with the lowest (best) p-value if it's under the threshold
        if best_pval < threshold_in:
            best_feature = new_pval.idxmin()
            included.append(best_feature)
            changed = True
            if verbose:
                print('Add  {:30} with p-value {:.6}'.format(best_feature, best_pval))

        # backward step: drop any feature that became insignificant after the forward step
        model = sm.OLS(y, sm.add_constant(pd.DataFrame(X[included]))).fit()
        # use all coefs except intercept
        pvalues = model.pvalues.iloc[1:]
        worst_pval = pvalues.max()  # null if pvalues is empty
        # if a feature's p-value exceeds the upper threshold, drop it
        if worst_pval > threshold_out:
            changed = True
            worst_feature = pvalues.idxmax()
            included.remove(worst_feature)
            if verbose:
                print('Drop {:30} with p-value {:.6}'.format(worst_feature, worst_pval))

        if not changed:
            break

    return included

In [ ]:
selected_features = stepwise_selection(X_train_encoded_poly_scaled, y_train, threshold_in=0.01, threshold_out=0.05, verbose=True)

X_train_final = X_train_encoded_poly_scaled[selected_features]
X_test_final = X_test_encoded_poly_scaled[selected_features]

# Linear Regression

## Using PyCaret to provide some guidance as to which model to use after implementing the necessary preprocessing steps

In [ ]:
# from pycaret import *

# exp = setup(
#     data=df,
#     target="Life_expectancy",
#     ignore_features=[
#         "Unnamed: 0",
#         "Country",
#     ], 
#     train_size=0.8,  
#     session_id=42,  
#     # Preprocessing pipeline setup:
#     categorical_features=["Region"],  # One-hot encoding
#     polynomial_features=True,  
#     polynomial_degree=2,
#     normalize=True,  # Standard scaling
#     normalize_method="zscore",  # Standardized z-scores (StandardScaler)
#     feature_selection=True,  
#     feature_selection_method="classic",  # Sequential / stepwise selection
#     n_features_to_select=0.2,  # Fraction or number of features to keep
# )

In [ ]:
# # 2. Train, evaluate, and rank candidate models
# best_models = compare_models(
#     include=["lr", "ridge", "lasso", "rf", "et", "gbr", "xgboost"],
#     sort="RMSE",  
# )

## Manually training Linear Regression Models

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": RidgeCV(
        alphas=[0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0], cv=5
    ),
    "Lasso Regression": LassoCV(cv=5, max_iter=10000, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
}


results = []

for name, model in models.items():

    model.fit(X_train_final, y_train)

    # Predict on train and test
    y_train_pred = model.predict(X_train_final)
    y_test_pred = model.predict(X_test_final)

    # Calculate metrics
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)

    train_rmse = root_mean_squared_error(y_train, y_train_pred)
    test_rmse = root_mean_squared_error(y_test, y_test_pred)

    # Store metrics in a dictionary
    results.append(
        {
            "Model": name,
            "Train R²": round(train_r2, 4),
            "Test R²": round(test_r2, 4),
            "Train MSE": round(train_mse, 4),
            "Test MSE": round(test_mse, 4),
            "Train RMSE": round(train_rmse, 4),
            "Test RMSE": round(test_rmse, 4),
        }
    )

# 3. Convert results list into a comparison DataFrame
results_df = pd.DataFrame(results)

# Sort by Test RMSE (lowest/best first)
results_df = results_df.sort_values(by="Test RMSE").reset_index(drop=True)

# Display the summary table
print("Model Performance Summary:\n")
display(results_df)